# Milestone 1 — EDA, Text Processing & Baseline Similarity Metrics



In [2]:
import string
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


## Q1 

In [3]:
answer_counts = train["answer"].value_counts()
print(answer_counts)

most_frequent = answer_counts.max()
least_frequent = answer_counts.min()

print("\nMost frequent count:", most_frequent)
print("Least frequent count:", least_frequent)
print("Sum:", most_frequent + least_frequent)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Most frequent count: 490
Least frequent count: 324
Sum: 814


## Q2 

In [4]:
def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return text.split()

all_words = set()
for prompt in train["prompt"]:
    all_words.update(clean_text(prompt))

print("Vocabulary size:", len(all_words))

Vocabulary size: 859


## Q3 

In [5]:
row1 = train[train["id"] == 1].iloc[0]
cleaned_words = clean_text(row1["prompt"])

filtered_words = [w for w in cleaned_words if w not in ENGLISH_STOP_WORDS]

print("Words remaining:", len(filtered_words))
print(filtered_words)

Words remaining: 13
['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']


## Q4



In [6]:
combined_texts = (
    train["prompt"] + " " +
    train["A"] + " " +
    train["B"] + " " +
    train["C"] + " " +
    train["D"] + " " +
    train["E"]
)

vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(combined_texts)

print("Number of feature columns:", len(vectorizer.get_feature_names_out()))

Number of feature columns: 2762


## Q5 

In [7]:
prompt_vec = vectorizer.transform([row1["prompt"]])
option_a_vec = vectorizer.transform([row1["A"]])

similarity = cosine_similarity(prompt_vec, option_a_vec)[0][0]
print(f"Cosine similarity (Prompt vs Option A), Row ID 1: {similarity:.4f}")

Cosine similarity (Prompt vs Option A), Row ID 1: 0.2720


## Q6 



In [8]:
options = ["A", "B", "C", "D", "E"]

# Vectorize every prompt and option column once (much faster than transforming row-by-row)
prompt_matrix = vectorizer.transform(train["prompt"])
option_matrices = {opt: vectorizer.transform(train[opt]) for opt in options}

correct_count = 0
for i in range(len(train)):
    sims = {
        opt: cosine_similarity(prompt_matrix[i], option_matrices[opt][i])[0][0]
        for opt in options
    }
    predicted = max(sims, key=sims.get)
    if predicted == train.iloc[i]["answer"]:
        correct_count += 1

accuracy_pct = (correct_count / len(train)) * 100
print(f"Percentage where highest similarity matches correct answer: {accuracy_pct:.2f}%")

Percentage where highest similarity matches correct answer: 13.55%


## Q7 

In [9]:
def apk(actual, predicted, k=3):
    predicted = predicted[:k]
    for i, p in enumerate(predicted):
        if p == actual:
            return 1.0 / (i + 1)
    return 0.0

score_q7 = apk("C", ["C", "A", "B"])
print("MAP@3:", score_q7)

MAP@3: 1.0


## Q8

In [10]:
score_q8 = apk("B", ["D", "B", "E"])
print("MAP@3:", score_q8)

MAP@3: 0.5


## Q9

In [11]:
top3_baseline = list(answer_counts.index[:3])
print("Baseline top 3 predictions:", top3_baseline)

baseline_scores = [apk(actual, top3_baseline) for actual in train["answer"]]
baseline_map3 = sum(baseline_scores) / len(baseline_scores)

print(f"Majority Class Baseline MAP@3: {baseline_map3:.5f}")

Baseline top 3 predictions: ['B', 'C', 'A']
Majority Class Baseline MAP@3: 0.42125


## Q10 

In [12]:
def get_top3_tfidf(i):
    sims = {
        opt: cosine_similarity(prompt_matrix[i], option_matrices[opt][i])[0][0]
        for opt in options
    }
    return sorted(sims, key=sims.get, reverse=True)[:3]

tfidf_scores = []
for i in range(len(train)):
    top3 = get_top3_tfidf(i)
    tfidf_scores.append(apk(train.iloc[i]["answer"], top3))

tfidf_map3 = sum(tfidf_scores) / len(tfidf_scores)
print(f"TF-IDF Pipeline MAP@3: {tfidf_map3:.6f}")

TF-IDF Pipeline MAP@3: 0.296167
